In [21]:
# Cell 1: Setup Walk-Forward Backtesting (FIXED VERSION)
import sys
sys.path.append('../')

# Force reload to get all fixes
import importlib
if 'utils.backtest_utils' in sys.modules:
    importlib.reload(sys.modules['utils.backtest_utils'])

from utils.backtest_utils import *
import matplotlib.pyplot as plt

print("🔄 Walk-Forward Backtesting (No Data Leakage) - FIXED")
print("="*50)

data_manager = BacktestDataManager()
backtester = WalkForwardBacktester(data_manager)

# Cell 2: Run Backtesting for All Symbols (with limited output)
all_results = {}

class LimitedOutput:
    def __init__(self, max_lines=100):
        self.max_lines = max_lines
        self.line_count = 0
        self.original_stdout = sys.stdout
    def write(self, text):
        if self.line_count < self.max_lines:
            self.original_stdout.write(text)
            if '\n' in text:
                self.line_count += text.count('\n')
        elif self.line_count == self.max_lines:
            self.original_stdout.write("... (detailed output truncated for brevity) ...\n")
            self.line_count += 1
    def flush(self):
        self.original_stdout.flush()

# Limit output during backtesting
limited_out = LimitedOutput(50)
original_stdout = sys.stdout

for symbol in SYMBOLS:
    print(f"\n🎯 Testing {symbol}...")
    sys.stdout = limited_out  # Limit detailed output
    limited_out.line_count = 0  # Reset counter

    # Diagnostics: track skipped periods
    try:
        # Modify run_walk_forward_backtest to return (results, skipped_periods) if diagnostics=True
        # If not implemented, fallback to old behavior
        try:
            results, skipped_periods = backtester.run_walk_forward_backtest(symbol, diagnostics=True)
        except TypeError:
            results = backtester.run_walk_forward_backtest(symbol)
            skipped_periods = []
        sys.stdout = original_stdout  # Restore output

        all_results[symbol] = results

        valid_periods = len(results) if results else 0
        print(f"🔎 {symbol}: Valid periods processed: {valid_periods}")
        if skipped_periods:
            print(f"⚠️ {symbol}: Skipped periods due to insufficient data:")
            for reason in skipped_periods:
                print(f"   - {reason}")

        if results:
            avg_return = np.mean([r['total_return'] for r in results])
            print(f"✅ {symbol} Average Return: {avg_return:.2f}%")
        else:
            print(f"❌ {symbol}: No results")

    except Exception as e:
        sys.stdout = original_stdout  # Restore output on error
        print(f"❌ {symbol} Error: {e}")

# Cell 3: Aggregate Results and Analysis
def analyze_backtest_results():
    """Analyze walk-forward backtest results"""
    
    if not all_results:
        print("❌ No backtest results to analyze")
        return
    
    print("\n📊 WALK-FORWARD BACKTEST RESULTS")
    print("="*60)
    
    # Summary by symbol
    for symbol, results in all_results.items():
        if not results:
            continue
        returns = [r['total_return'] for r in results]
        trades = [r['num_trades'] for r in results]
        print(f"\n{symbol}:")
        print(f"  Periods tested: {len(returns)}")
        print(f"  Average return: {np.mean(returns):.2f}%")
        print(f"  Std deviation: {np.std(returns):.2f}%")
        print(f"  Best period: {max(returns):.2f}%")
        print(f"  Worst period: {min(returns):.2f}%")
        print(f"  Win rate: {len([r for r in returns if r > 0]) / len(returns) * 100:.1f}%")
        print(f"  Avg trades/period: {np.mean(trades):.1f}")
    
    # Overall portfolio performance
    all_returns = []
    for results in all_results.values():
        all_returns.extend([r['total_return'] for r in results])
    
    if all_returns:
        print(f"\n🏆 OVERALL PORTFOLIO:")
        print(f"  Total periods: {len(all_returns)}")
        print(f"  Average return: {np.mean(all_returns):.2f}%")
        print(f"  Sharpe-like ratio: {np.mean(all_returns) / np.std(all_returns):.2f}")
        print(f"  Win rate: {len([r for r in all_returns if r > 0]) / len(all_returns) * 100:.1f}%")

analyze_backtest_results()

🧠 Loading FinBERT...
🔄 Walk-Forward Backtesting (No Data Leakage) - FIXED

🎯 Testing AAPL...

🔄 Walk-Forward Backtesting: AAPL
  Total months available: 69
  Window size needed: 33
  Possible periods: 37

📅 Period 1:
  Train: 2020-01-01 to 2022-01-20
  Val:   2022-01-20 to 2022-06-19
  Test:  2022-06-19 to 2022-09-17
    Data sizes - Train: 517, Val: 103, Test: 61
    Sequences created - Train: 507, Val: 93, Test: 51
  📊 Test Return: 0.00%

📅 Period 2:
  Train: 2020-03-31 to 2022-04-20
  Val:   2022-04-20 to 2022-09-17
  Test:  2022-09-17 to 2022-12-16
    Data sizes - Train: 518, Val: 103, Test: 63
    Sequences created - Train: 508, Val: 93, Test: 53
🔄 Walk-Forward Backtesting (No Data Leakage) - FIXED

🎯 Testing AAPL...

🔄 Walk-Forward Backtesting: AAPL
  Total months available: 69
  Window size needed: 33
  Possible periods: 37

📅 Period 1:
  Train: 2020-01-01 to 2022-01-20
  Val:   2022-01-20 to 2022-06-19
  Test:  2022-06-19 to 2022-09-17
    Data sizes - Train: 517, Val: 103, Te

In [22]:
# Debug the portfolio calculation
print("🔍 Debugging Portfolio Calculation")

import importlib
if 'utils.backtest_utils' in sys.modules:
    importlib.reload(sys.modules['utils.backtest_utils'])

from utils.backtest_utils import *

# Create a simple test of the evaluation function
data_manager = BacktestDataManager()
backtester = WalkForwardBacktester(data_manager)

# Get test data for one period
symbol = 'AAPL'
val_end = datetime.strptime('2022-08-01', "%Y-%m-%d")
test_end = datetime.strptime('2022-09-01', "%Y-%m-%d")

X_test, y_test, test_df = backtester.prepare_data_for_period(symbol, val_end, test_end)

if X_test is not None:
    # Create scaled sequences
    scaler = MinMaxScaler()
    X_test_scaled = scaler.fit_transform(X_test)
    X_test_seq, y_test_seq = backtester.create_sequences(X_test_scaled, y_test)
    
    # Create a simple test model that returns predictable values
    class TestModel:
        def eval(self): pass
        def __call__(self, x):
            import torch
            # Return alternating high/low predictions to force trading
            batch_size = x.size(0)
            values = []
            for i in range(batch_size):
                if i % 2 == 0:
                    values.append(200.0)  # High prediction (buy signal)
                else:
                    values.append(100.0)  # Low prediction (sell signal)
            return torch.tensor(values).unsqueeze(1).float()
    
    test_model = TestModel()
    
    # Manual evaluation with debug prints
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_test_tensor = torch.FloatTensor(X_test_seq).to(device)
    
    with torch.no_grad():
        predictions = test_model(X_test_tensor).squeeze().cpu().numpy()
    
    lookback = 10
    df_subset = test_df.iloc[lookback:].reset_index(drop=True)
    actual_prices = df_subset['close'].values
    
    min_len = min(len(predictions), len(actual_prices))
    predictions = predictions[:min_len]
    actual_prices = actual_prices[:min_len]
    
    print(f"Predictions: {predictions[:5]}")
    print(f"Actual prices: {actual_prices[:5]}")
    
    # Manual portfolio simulation with debug
    portfolio_value = 10000
    position = 0
    trade_log = []
    
    for i in range(min_len - 1):
        current_price = actual_prices[i]
        next_day_prediction = predictions[i]
        
        predicted_return = (next_day_prediction - current_price) / current_price
        
        if predicted_return > 0.001 and position == 0:  # Buy
            shares = portfolio_value / current_price
            position = shares
            portfolio_value = 0
            trade_log.append(f"Day {i}: BUY {shares:.2f} shares at ${current_price:.2f} (pred return: {predicted_return:.3f})")
            
        elif predicted_return < -0.001 and position > 0:  # Sell
            cash = position * current_price
            trade_log.append(f"Day {i}: SELL {position:.2f} shares at ${current_price:.2f} for ${cash:.2f}")
            portfolio_value = cash
            position = 0
    
    # Final value
    if position > 0:
        final_cash = position * actual_prices[-1]
        trade_log.append(f"Final: SELL {position:.2f} shares at ${actual_prices[-1]:.2f} for ${final_cash:.2f}")
        portfolio_value = final_cash
    
    print(f"\nTrade log:")
    for log in trade_log[:10]:  # Show first 10 trades
        print(f"  {log}")
    
    total_return = (portfolio_value - 10000) / 10000 * 100
    print(f"\nFinal portfolio value: ${portfolio_value:.2f}")
    print(f"Total return: {total_return:.2f}%")
    
else:
    print("❌ No test data available")

🔍 Debugging Portfolio Calculation
🧠 Loading FinBERT...
Predictions: [200. 100. 200. 100. 200.]
Actual prices: [170.49861145 170.34107971 171.8374939  171.44366455 168.85455322]

Trade log:
  Day 0: BUY 58.65 shares at $170.50 (pred return: 0.173)
  Day 1: SELL 58.65 shares at $170.34 for $9990.76
  Day 2: BUY 58.14 shares at $171.84 (pred return: 0.164)
  Day 3: SELL 58.14 shares at $171.44 for $9967.86
  Day 4: BUY 59.03 shares at $168.85 (pred return: 0.184)
  Day 5: SELL 59.03 shares at $164.97 for $9738.31
  Day 6: BUY 59.15 shares at $164.63 (pred return: 0.215)
  Day 7: SELL 59.15 shares at $164.93 for $9755.78
  Day 8: BUY 58.28 shares at $167.39 (pred return: 0.195)
  Day 9: SELL 58.28 shares at $161.08 for $9387.99

Final portfolio value: $9244.31
Total return: -7.56%
Predictions: [200. 100. 200. 100. 200.]
Actual prices: [170.49861145 170.34107971 171.8374939  171.44366455 168.85455322]

Trade log:
  Day 0: BUY 58.65 shares at $170.50 (pred return: 0.173)
  Day 1: SELL 58.65 